# Kaggriculture: V13 vs V14.4 — Reproducible Local Benchmark and Strategy Review

This notebook documents a balanced 20-match local comparison and turns the result into a community-reviewable strategy report.

**Result:** the head-to-head contest was a statistical draw: V13 won 1, V14.4 won 1, and 18 matches tied. Both completed all games, with mean and median V14.4 margin equal to 0. Under a conservative incumbent tie-break, **V13 remains the champion**. The reviewed public-score snapshot also favored V13 (555.9 vs 528.8), but that telemetry is observational rather than a controlled causal comparison.

No cell submits to Kaggle, accesses credentials, or requires private workspace files.

## 1. Kaggriculture Problem Summary

Kaggriculture is a 720-turn, two-player farming economy with simultaneous decisions, stochastic events, and partial observability.

- **Objective:** finish with more money than the opponent.
- **Production:** buy seeds, plant, water, fertilize, harvest, transport, and sell crops; build facilities and manage livestock production.
- **Labor:** coordinate the farmer and hired hands across movement and tile actions.
- **Economics:** protect cash reserves, storage capacity, lifecycle payback, and terminal liquidation.
- **Interaction:** players share market supply and dynamic prices, so order timing and product concentration matter.
- **Information:** private inventories and simultaneous actions prevent exact opponent-state reconstruction.
- **Planning challenge:** tactical movement, daily care, long-horizon investments, market contention, and the turn-720 deadline must all be handled under strict runtime limits.

## 2. Runtime and Space Complexity

Let $T=720$ turns, $U$ controlled units, $A$ legal actions per unit, $K$ retained macro-actions, $D$ planning depth, and $N$ simulations.

A naive joint-action tree grows roughly as $O((A^U)^T)$ before adding ordered market lists, opponent actions, chance outcomes, and information sets. Even a movement-only approximation yields about $10^{1007}$ joint trajectories across 720 turns. Full brute force is impossible.

A practical planner therefore needs:

- legal macro-action generation rather than raw action products;
- global worker assignment rather than independent branching;
- top-$K$ market bundles;
- hard depth, node, memory, and latency budgets;
- dominance pruning and transposition caching;
- sampled uncertainty rather than exhaustive chance expansion;
- a deterministic fail-closed fallback.

V13's submitted middle-game selector is much smaller: at most four market-order permutations and 48 deterministic one-step scoring selections. Its per-turn search cost is approximately $O(NKL)$ time for market-list length $L\le10$, with $O(K)$ node storage. This is a bounded market-order bandit, not a full multi-turn MCTS.

In [ ]:
# Lightweight empirical resource profile of the analysis path.
# This cell runs after the benchmark-result table has been created in Section 6.
if "matches" in globals():
    tracemalloc.start()
    started = time.perf_counter()
    for _ in range(1000):
        _ = matches.groupby(["seed", "v14_seat"])["v14_margin"].sum()
    analysis_ms = (time.perf_counter() - started) * 1000
    _, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    complexity_profile = pd.DataFrame([{
        "analysis_iterations": 1000,
        "elapsed_ms": round(analysis_ms, 2),
        "peak_memory_kib": round(peak_bytes / 1024, 2),
        "v13_macro_candidates_max": 4,
        "v13_selection_iterations": 48,
        "market_orders_max": 10,
    }])
    display(complexity_profile)
else:
    print("Run Sections 3–6 first, then rerun this optional empirical profile.")

## 3. Configure Kaggle-Compatible Environment

The verified schedule uses ten deterministic seeds and both seats, yielding exactly 20 games. Paths are relative and may be overridden with environment variables. Set `RUN_MATCHES=True` only when the two archives and `kaggle-environments` are available; otherwise this notebook analyzes the embedded verified results.

In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import io
import json
import os
import sys
import tarfile
import tempfile
import time
import tracemalloc
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

SEEDS = [11, 23, 37, 41, 53, 67, 79, 83, 97, 109]
EXPECTED_GAMES = 20
REQUIRED_SUBMISSION_WINS = 13
RUN_MATCHES = os.environ.get("RUN_KAGGRICULTURE_MATCHES", "0") == "1"
ON_KAGGLE = Path("/kaggle").exists()
BASE = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
OUTPUT_DIR = Path(os.environ.get("KAGGRICULTURE_OUTPUT", str(BASE / "kaggriculture_v13_v14_4")))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({"on_kaggle": ON_KAGGLE, "run_matches": RUN_MATCHES, "output": str(OUTPUT_DIR)})

## 4. Load Kaggriculture Agents V13 and V14.4

The optional loader accepts the exact `submission.tar.gz` format used by the competition. It validates archive membership, extracts only `main.py` and `agent_payload.zip` into a content-addressed temporary directory, and checks that the bootstrap exports a callable `agent`. Attach the two archives as a Kaggle Dataset or place them beside the notebook to reproduce matches.

In [ ]:
def find_archive(version: str) -> Path | None:
    candidates = [
        Path(f"MyAgents/{version}/submission.tar.gz"),
        Path(f"../MyAgents/{version}/submission.tar.gz"),
        Path(f"/kaggle/input/kaggriculture-agents/{version}/submission.tar.gz"),
        Path(f"{version}/submission.tar.gz"),
    ]
    return next((path for path in candidates if path.is_file()), None)


def load_submission_archive(path: Path):
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    root = Path(tempfile.gettempdir()) / f"kaggriculture_{digest[:20]}"
    root.mkdir(parents=True, exist_ok=True)
    with tarfile.open(path, "r:gz") as archive:
        members = archive.getmembers()
        names = {member.name.replace("\\", "/") for member in members}
        if names != {"main.py", "agent_payload.zip"} or any(not member.isfile() for member in members):
            raise ValueError("Unexpected submission archive structure")
        for member in members:
            handle = archive.extractfile(member)
            if handle is None:
                raise ValueError(f"Could not read {member.name}")
            (root / member.name).write_bytes(handle.read())
    spec = importlib.util.spec_from_file_location(f"submitted_{digest[:20]}", root / "main.py")
    if spec is None or spec.loader is None:
        raise RuntimeError("Cannot load agent bootstrap")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    if not callable(getattr(module, "agent", None)):
        raise TypeError("Archive does not expose callable agent")
    return module.agent, digest


archives = {version: find_archive(version) for version in ("v13", "v14.4")}
print({version: str(path) if path else "not attached" for version, path in archives.items()})

## 5. Build a Reproducible Local Match Harness

Each seed is played twice with reversed seats. The harness records final money, status, duration, and replay JSON. Exceptions are captured as diagnostics rather than silently discarded.

In [ ]:
def final_money(final_agent_state: dict, seat: int) -> float:
    farms = (final_agent_state.get("observation") or {}).get("farms") or []
    return float((farms[seat] or {}).get("money", 0)) if seat < len(farms) else float(final_agent_state.get("reward", 0))


def run_match(v13_agent, v14_agent, seed: int, v14_seat: int, replay_dir: Path) -> dict:
    from kaggle_environments import make

    agents = [v13_agent, v14_agent] if v14_seat == 1 else [v14_agent, v13_agent]
    started = time.perf_counter()
    try:
        env = make("kaggriculture", configuration={"episodeSteps": 720, "seed": seed}, debug=False)
        env.run(agents)
        replay = env.toJSON()
        final = replay["steps"][-1]
        v13_seat = 1 - v14_seat
        v14_money = final_money(final[v14_seat], v14_seat)
        v13_money = final_money(final[v13_seat], v13_seat)
        margin = v14_money - v13_money
        replay_dir.mkdir(parents=True, exist_ok=True)
        replay_path = replay_dir / f"seed{seed}-v14seat{v14_seat}.json"
        replay_path.write_text(json.dumps(replay), encoding="utf-8")
        return {
            "seed": seed, "v14_seat": v14_seat, "v13_money": v13_money,
            "v14_money": v14_money, "v14_margin": margin,
            "winner": "v14.4" if margin > 0 else "v13" if margin < 0 else "tie",
            "v13_status": str(final[v13_seat].get("status", "UNKNOWN")),
            "v14_status": str(final[v14_seat].get("status", "UNKNOWN")),
            "elapsed_seconds": time.perf_counter() - started,
            "replay": str(replay_path), "diagnostic": "",
        }
    except Exception as exc:
        return {
            "seed": seed, "v14_seat": v14_seat, "v13_money": None, "v14_money": None,
            "v14_margin": None, "winner": "error", "v13_status": "UNKNOWN",
            "v14_status": "ERROR", "elapsed_seconds": time.perf_counter() - started,
            "replay": "", "diagnostic": f"{type(exc).__name__}: {exc}",
        }

## 6. Run 20 Balanced Matches

The embedded rows below are the exact results generated locally from the hash-verified V13 and V14.4 archives. When both archives are attached and `RUN_KAGGRICULTURE_MATCHES=1`, the cell reruns the same schedule instead.

In [ ]:
embedded = [
    (11, 0, 25439, 25439), (11, 1, 25439, 25439),
    (23, 0, 23616, 21716), (23, 1, 21716, 23616),
    (37, 0, 25711, 25711), (37, 1, 25711, 25711),
    (41, 0, 24923, 24923), (41, 1, 24923, 24923),
    (53, 0, 24934, 24934), (53, 1, 24934, 24934),
    (67, 0, 25783, 25783), (67, 1, 25783, 25783),
    (79, 0, 24218, 24218), (79, 1, 24218, 24218),
    (83, 0, 25289, 25289), (83, 1, 25289, 25289),
    (97, 0, 24395, 24395), (97, 1, 24395, 24395),
    (109, 0, 24716, 24716), (109, 1, 24716, 24716),
]

if RUN_MATCHES:
    if not all(archives.values()):
        raise FileNotFoundError("Attach both V13 and V14.4 submission archives before rerunning")
    v13_agent, v13_hash = load_submission_archive(archives["v13"])
    v14_agent, v14_hash = load_submission_archive(archives["v14.4"])
    rows = [run_match(v13_agent, v14_agent, seed, seat, OUTPUT_DIR / "replays") for seed in SEEDS for seat in (0, 1)]
    matches = pd.DataFrame(rows)
else:
    matches = pd.DataFrame(embedded, columns=["seed", "v14_seat", "v13_money", "v14_money"])
    matches["v14_margin"] = matches["v14_money"] - matches["v13_money"]
    matches["winner"] = matches["v14_margin"].map(lambda x: "v14.4" if x > 0 else "v13" if x < 0 else "tie")
    matches["v13_status"] = matches["v14_status"] = "DONE"
    matches["elapsed_seconds"] = pd.NA
    matches["replay"] = "embedded verified result"
    matches["diagnostic"] = ""

assert len(matches) == EXPECTED_GAMES
assert matches["winner"].value_counts().to_dict() == {"tie": 18, "v13": 1, "v14.4": 1}
matches

## 7. Determine the Winning Agent

Selection order is match wins, aggregate margin, failures, then runtime. Every competitive tie retains the incumbent to avoid promoting an unproven change. With 1 win each, zero aggregate margin, and no failures, V13 is retained—not declared intrinsically superior. Neither agent clears the separate 13-of-20 submission challenge.

In [ ]:
counts = matches["winner"].value_counts()
summary = pd.DataFrame({
    "agent": ["v13", "v14.4"],
    "wins": [int(counts.get("v13", 0)), int(counts.get("v14.4", 0))],
    "losses": [int(counts.get("v14.4", 0)), int(counts.get("v13", 0))],
    "draws": [int(counts.get("tie", 0))] * 2,
    "mean_margin": [-matches["v14_margin"].mean(), matches["v14_margin"].mean()],
    "median_margin": [-matches["v14_margin"].median(), matches["v14_margin"].median()],
    "failures": [int((matches["v13_status"] != "DONE").sum()), int((matches["v14_status"] != "DONE").sum())],
})
LOCAL_DECISION = "v13 (retained incumbent after a drawn challenge)"
print(summary.to_string(index=False))
print(f"\nDecision: {LOCAL_DECISION}")
print("Uncertainty: 18/20 ties provide little power to distinguish these closely related policies.")

## 8. V13 Key Strategy

Source audit of the frozen artifact found this hierarchy:

1. **Trusted core:** exact V12 supplies the complete action bundle and remains the fallback.
2. **Phase routing:** exact V12 is used directly on turns 1–5 and 700–720.
3. **Middle-game market selection:** on turns 6–699, V13 keeps every farmer and hand action unchanged, filters the baseline market list, and compares at most four orderings: exact, cashflow-first, survival-first, and growth-first.
4. **Bounded scoring:** 48 deterministic UCT-style selections use one-step hand-scored outcomes under four fixed price multipliers.
5. **Inherited economy:** crop diversification, MELON production, GOOSE feed/care, mixed GOOSE+COW livestock, six hired hands, same-turn target claims, batch worker assignment, and late seed-purchase cutoffs.
6. **Risk control:** structural validation and exception fallback to exact V12.

**Strengths:** deterministic, fast, dependency-light, interpretable, hash-verifiable, and resistant to catastrophic search failures.

**Limitations:** no multi-turn transition tree, no explicit opponent-action model, no alternative worker actions in search, hand-tuned utility values, only four price scenarios, and no returned live diagnostics. The 18 ties show that V14.4's repair/liquidation overlay rarely changed final money on this schedule. Seed 23 was the only divergence, and its sign reversed with seat, so it is not evidence of a robust overlay advantage.

In [ ]:
# Optional direct decision-latency probe when the V13 archive is attached.
if archives["v13"]:
    v13_probe, _ = load_submission_archive(archives["v13"])
    tiles = [[None if x < 5 and y < 5 else "LOCKED" for x in range(10)] for y in range(10)]
    farm = {"money": 3000, "tiles": tiles, "farmer": [4, 4], "hands": [], "unlocked_quadrants": ["NW"], "hires_today": 0}
    observation = {
        "player": 0, "step": 0, "day": 0, "hour": 0, "farms": [farm, farm],
        "private": {"shed": {}, "seeds": {}, "inventories": [{}]},
        "market": {"inventory": {}, "prices": {}}, "town": {"unlocked_shops": []},
    }
    samples = []
    tracemalloc.start()
    for _ in range(100):
        started = time.perf_counter()
        action = v13_probe(observation)
        samples.append((time.perf_counter() - started) * 1000)
    _, agent_peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    assert set(action) == {"farmer", "hands", "market"}
    print({"mean_decision_ms": sum(samples) / len(samples), "max_decision_ms": max(samples), "peak_kib": agent_peak / 1024})
else:
    print("Attach V13 to run the direct latency probe; benchmark conclusions remain available from embedded results.")

## 9. Match Results and Failure Modes

The plots emphasize the unusually high tie rate and the single seat-sensitive divergence. Important failure questions for future replay analysis include invalid or ineffective actions, idle workers, duplicate targets, crop or livestock loss, inventory overflow, late investments, missed liquidation, and decision-time spikes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
order = ["v13", "tie", "v14.4"]
counts.reindex(order, fill_value=0).plot.bar(ax=axes[0], color=["#3b82f6", "#94a3b8", "#f59e0b"], title="Match outcomes")
axes[0].set_ylabel("matches")

matches.assign(game=range(1, len(matches) + 1)).plot(x="game", y="v14_margin", marker="o", ax=axes[1], title="V14.4 margin by match", legend=False)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_ylabel("final-money margin")

role = matches.groupby("v14_seat")["v14_margin"].agg(["mean", "median", "min", "max"])
role["mean"].plot.bar(ax=axes[2], color="#8b5cf6", title="Mean V14.4 margin by seat")
axes[2].axhline(0, color="black", linewidth=1)
axes[2].set_xlabel("V14.4 seat")
plt.tight_layout()
plot_path = OUTPUT_DIR / "v13_v14_4_benchmark.png"
fig.savefig(plot_path, dpi=160, bbox_inches="tight")
plt.show()

print("Status failures:", int(((matches.v13_status != "DONE") | (matches.v14_status != "DONE")).sum()))
print("Only nonzero-margin rows:")
display(matches.loc[matches.v14_margin != 0, ["seed", "v14_seat", "v13_money", "v14_money", "v14_margin", "winner"]])

In [ ]:
timeline = pd.DataFrame({
    "game": range(1, len(matches) + 1),
    "V13 cumulative wins": (matches.winner == "v13").cumsum(),
    "V14.4 cumulative wins": (matches.winner == "v14.4").cumsum(),
})
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
timeline.plot(x="game", y=["V13 cumulative wins", "V14.4 cumulative wins"], drawstyle="steps-post", ax=axes[0], title="Cumulative decisive wins")
matches.boxplot(column="v14_margin", by="v14_seat", ax=axes[1])
axes[1].set_title("Margin distribution by V14.4 seat")
axes[1].set_xlabel("V14.4 seat")
axes[1].set_ylabel("V14.4 final-money margin")
plt.suptitle("")
plt.tight_layout()
plt.show()
role

## 10. Next Improvement Strategy

The next candidate should change one attributable mechanism and preserve V13 as fail-closed fallback.

| Priority | Experiment | Expected impact | Cost | Timeout risk |
|---:|---|---:|---:|---:|
| 1 | Replay first-divergence and ineffective-action extraction | High | Medium | Low |
| 2 | Global worker-to-task matching with exclusive claims | High | Medium | Low |
| 3 | Lifecycle-aware crop/livestock value and terminal payback | High | Medium | Low |
| 4 | Adaptive node/time allocation for meaningful market choices | Medium | Medium | Medium |
| 5 | Canonical state cache and dominance pruning | Medium | Medium | Low |
| 6 | Observation-only opponent regime features | Medium | Medium | Low |
| 7 | Tune scoring coefficients on frozen development seeds | Medium | Low | Low |
| 8 | Learned leaf-state value model inside bounded search | Potentially high | High | Medium |

Recommended sequence:

1. Download and normalize official replays.
2. Explain why seed 23 diverged and why the sign followed seat.
3. Add permanent difficult-state regression fixtures.
4. Implement global assignment without changing economic policy.
5. Add lifecycle economics and terminal liquidation as separate candidates.
6. Use 20-game champion challenges only as final direct gates; retain larger multi-opponent validation for generalization.
7. Add learned evaluation only after deterministic macro-actions, pruning, cache telemetry, and latency fallback are reliable.

## 11. Kaggriculture Community Feedback Request

### Help us break a very stable policy tie

We ran a reproducible 20-game local comparison of V13 and V14.4 using ten fixed seeds and both seats. The outcome was **V13 1 win, V14.4 1 win, and 18 ties**, with 100% completion and zero aggregate margin. We conservatively retained V13 as incumbent. V14.4 adds state-derived invariant repair and terminal liquidation, but it changed final money only on seed 23—and the sign reversed with seat.

We would value focused community suggestions:

1. Which game states most reliably expose weak task assignment or terminal liquidation?
2. What opponent archetypes should a small but representative validation league contain?
3. Which replay features best distinguish an action that merely diverges from one that improves win probability?
4. Are there overlooked crop/livestock lifecycle interactions or shared-market effects worth modeling first?
5. What profiling approach best measures per-decision latency and memory inside Kaggle's agent harness?
6. How would you allocate a strict search budget between market bundles, worker assignment, and uncertainty sampling?
7. Which deterministic caching or pruning techniques transfer well to this partially observable setting?
8. How can we make a direct 20-game champion test more discriminative without sacrificing seat balance and reproducibility?

Please suggest mechanisms and representative scenarios rather than exact copied action schedules. We want improvements that generalize, remain legal and fast, and can be validated with transparent paired experiments.

## 12. Export Results and Kaggle-Ready Artifacts

This final cell exports tables and a compact summary. It also checks that recorded replay references and configuration do not leak Windows absolute paths. Agent archives are optional inputs and are never repackaged or submitted here.

### Reproducibility and responsible sharing

- No credentials or submission API calls are present.
- Embedded rows are labeled as verified prior results; reruns require explicit opt-in.
- V13 is retained through a declared tie-break, not presented as conclusively superior.
- Public scores are timestamped observational evidence, not causal A/B results.
- Exact competitor schedules should not be copied; community evidence should become aggregate, testable hypotheses.

In [ ]:
matches_path = OUTPUT_DIR / "match_results.csv"
summary_path = OUTPUT_DIR / "aggregate_metrics.csv"
metadata_path = OUTPUT_DIR / "benchmark_metadata.json"

matches.to_csv(matches_path, index=False)
summary.to_csv(summary_path, index=False)
metadata = {
    "games": len(matches),
    "seeds": SEEDS,
    "seat_policy": "each seed in both V14.4 seats",
    "outcomes": matches.winner.value_counts().to_dict(),
    "mean_v14_margin": float(matches.v14_margin.mean()),
    "median_v14_margin": float(matches.v14_margin.median()),
    "completion_rate": float(((matches.v13_status == "DONE") & (matches.v14_status == "DONE")).mean()),
    "decision": LOCAL_DECISION,
    "submission_gate_required_wins": REQUIRED_SUBMISSION_WINS,
    "submission_performed": False,
}
metadata_path.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")

serialized = json.dumps(metadata) + matches.to_csv(index=False)
assert "C:\\" not in serialized and "c:\\" not in serialized
assert metadata["games"] == 20
assert metadata["outcomes"] == {"tie": 18, "v13": 1, "v14.4": 1}
print("Exported:", matches_path, summary_path, metadata_path, plot_path, sep="\n- ")